# Scanner de Temas Emergentes V001

| Celda | Que hace | Cuando |
|-------|----------|--------|
| 1 | Instala librerias | Solo la primera vez |
| 2 | Carga el codigo | SIEMPRE antes de 3 o 4 |
| 3 | Watchlist personal | ~45 seg |
| 4 | S&P 500 completo | ~5 min |

**Seguridad — credenciales:** antes de ejecutar, anade tus 4 secrets en el panel de Colab Secrets (icono de llave a la izquierda): `SCANNER_TOKEN`, `ANTHROPIC_KEY`, `TELEGRAM_TOKEN`, `TELEGRAM_CHAT_ID`. Activa "notebook access" en cada uno. **Nunca pegues una credencial como texto literal en una celda** — si el notebook se comparte o se sube a un gist, la credencial queda expuesta.

Novedades v10: RSI, MACD, ADX, Bollinger, CMF, OBV, ATR + Score de Confirmacion Tecnica (SCT)

Novedades v11 (correccion de inconsistencias matematicas):
- Objetivo acotado a R/B maximo 1:5 (antes usaba el maximo de 52 semanas directamente, inflando el R/B reportado)
- RIESGO (BAJO/MEDIO/ALTO) calculado en Python con formula exacta sobre SCT/RSI/CMF/ADX, ya no se delega a Claude
- Filtro ADX>=20 obligatorio en setups validos (descarta tendencias debiles)
- RANKING calculado en Python (SCT 40% + R/B normalizado 20% + sector 20% + CMF 20%)
- Objetivo escalonado: parcial (resistencia inmediata) y final (techo acotado)
- Distincion explicita Ruptura activa / Ruptura pendiente / Pullback MM50 / Pullback MM200
- Trigger concreto de reentrada en setups EXTENDIDOS (MM50 +-4% + RSI>55 + CMF>0.05)


Novedades v12 (punto 7):
- Earnings proximos detectados via yfinance para los setups finales
- Earnings <10 dias: el setup se descarta y se reemplaza por el siguiente del ranking
- Earnings 10-21 dias: el setup se mantiene pero el informe incluye aviso explicito de riesgo de gap
- Noticias recientes (hasta 3) incorporadas como contexto cuando aportan informacion relevante


In [ ]:
!pip install yfinance pandas requests beautifulsoup4 anthropic pytz httpx -q
print('OK librerias instaladas')

In [ ]:
# CELDA 2 — Ejecutar SIEMPRE antes de la 4
# =========================================================================
# P89 (19/09/2026) — Esta celda ya NO contiene el codigo del scanner.
#
# Antes duplicaba las ~5.200 lineas de scanner.py, y esa copia tenia que
# mantenerse identica linea por linea (paridad AST). Dos problemas:
#   1. 334.000 caracteres colgaban el editor de Colab.
#   2. Cada parche habia que aplicarlo DOS veces, y bastaba un despiste
#      para que el repo y el notebook divergieran (paso el 10/08).
#
# Ahora hay UNA sola fuente de verdad: scanner.py en el repo. Esta celda lo
# descarga y lo ejecuta, asi que el notebook nunca puede quedarse atras.
# =========================================================================
import os, requests

RAMA = 'main'   # cambiar solo para probar una rama distinta
URL  = f'https://raw.githubusercontent.com/Carolo-III/scanner-temas/{RAMA}/scanner.py'

# 1) Credenciales: de Colab Secrets al entorno, porque scanner.py las lee de
#    os.environ. Hay que hacerlo ANTES de ejecutarlo: las constantes del
#    modulo (GITHUB_TOKEN, ANTHROPIC_KEY...) se resuelven al importarse.
try:
    from google.colab import userdata
    for _s in ('SCANNER_TOKEN', 'ANTHROPIC_KEY', 'TELEGRAM_TOKEN',
               'TELEGRAM_CHAT_ID', 'FMP_KEY'):
        try:
            _v = userdata.get(_s)
            if _v:
                os.environ[_s] = _v
        except Exception as _e:
            print(f'  aviso: secreto {_s} no disponible en Colab ({type(_e).__name__})')
except ImportError:
    pass   # fuera de Colab se asume que ya estan en el entorno

# 2) Descarga
_r = requests.get(URL, timeout=60)
if _r.status_code != 200:
    raise RuntimeError(f'No se pudo descargar scanner.py: HTTP {_r.status_code}')
_src = _r.text
if 'def main(' not in _src:
    raise RuntimeError('scanner.py descargado pero no parece valido (falta main)')

# 3) Ejecucion en el espacio global del notebook, para que la Celda 4 vea las
#    funciones. Se cambia __name__ temporalmente: scanner.py termina con
#    "if __name__ == '__main__': main()" y en Colab __name__ YA es '__main__',
#    asi que sin esta guarda el pipeline entero arrancaria aqui solo.
_g = globals()
_nombre_previo = _g.get('__name__')
_g['__name__'] = 'scanner_remoto'
try:
    exec(compile(_src, 'scanner.py', 'exec'), _g)
finally:
    _g['__name__'] = _nombre_previo

print(f'OK scanner.py cargado desde {RAMA} ({len(_src):,} caracteres)')
_n = sum(1 for _v in _g.values() if type(_v).__name__ == 'function')
print(f'   {_n} funciones disponibles para la Celda 4')
print(f'   SCANNER_TOKEN {"OK" if GITHUB_TOKEN else "FALTA"} | ANTHROPIC_KEY {"OK" if ANTHROPIC_KEY else "FALTA"}')


In [ ]:
# CELDA 4 — S&P 500 completo (~5 min)
# CELDA 4 — S&P 500 completo (~5 min)
import warnings, json, pandas as pd
from datetime import datetime
import pytz
warnings.filterwarnings('ignore')

madrid = pytz.timezone('Europe/Madrid')

if not GITHUB_TOKEN or not ANTHROPIC_KEY:
    print('ERROR: Pon GITHUB_TOKEN y ANTHROPIC_KEY en la Celda 2')
else:
    print('SPY...')
    bc, bv, bh, bl = download_prices(['SPY'], period='1y')
    bs = bc['SPY'].squeeze()
    spy_ok, spy_score_continuo = spy_health(bs)  # P57: tupla (bool, float)

    try:
        spy_ret_1w = round((float(bs.iloc[-1]) / float(bs.iloc[-5])  - 1) * 100, 1)
        spy_ret_1m = round((float(bs.iloc[-1]) / float(bs.iloc[-20]) - 1) * 100, 1)
    except:
        spy_ret_1w = None; spy_ret_1m = None

    print('\n▸ Macro (VIX, DXY, US10Y)...')
    macro = get_macro_data()

    pt = list(set(t for grp in PERSONAL_WATCHLIST.values() for t in grp))
    # PUNTO 56 — espejo manual: put/call junto al bloque macro, ANTES de cualquier uso.
    putcall = get_putcall_cboe()
    if putcall:
        print('  Put/call (Cboe): ' + ' | '.join(f'{k}={v}' for k, v in sorted(putcall.items())))

    print('\n▸ Watchlist (' + str(len(pt)) + ' valores)...')
    cp, vp, hp, lp = download_prices(pt + ['SPY'], period='1y')
    pr  = analyze_universe(PERSONAL_WATCHLIST, bs, cp, vp, hp, lp, spy_healthy=spy_ok, spy_score=spy_score_continuo)
    pgs = calc_groups(pr, is_sp=False)
    print('  OK ' + str(len(pr)) + ' valores')
    # PUNTO 26 — nombrar a los tickers pedidos que no llegan al analisis
    faltan_wl = sorted(set(pt) - {r['ticker'] for r in pr})
    if faltan_wl: print('  Sin datos/analisis en watchlist: ' + ', '.join(faltan_wl))

    print('\n▸ Lista S&P 500...')
    url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv'
    df_sp = pd.read_csv(url)
    sbs = {}
    for _, row in df_sp.iterrows():
        tk = row['Symbol'].replace('.', '-')
        if tk not in pt:
            lb = SECTOR_LABELS.get(row['GICS Sector'], row['GICS Sector'])
            sbs.setdefault(lb, []).append(tk)

    sa = list(set(t for grp in sbs.values() for t in grp))

    # Ampliar con Nasdaq 100 y SOX
    nasdaq100_extra = [
        'ABNB','ADSK','ALGN','ASML','BIDU','BIIB',
        'CDNS','CHTR','CPRT','CTSH','DDOG','DLTR','DOCU','DXCM',
        'EA','EBAY','FAST','FSLR','FTNT','GFS',
        'IDXX','ILMN','KDP','KHC','LULU','MAR','MCHP','MDLZ',
        'MNST','MRNA','MRVL','NTES','ODFL','OKTA','ON','ORLY',
        'PANW','PAYX','PCAR','PDD','PYPL','QCOM','REGN','ROST',
        'SNPS','SWKS','TEAM','TMUS','TSCO',
        'TTD','TTWO','TXN','VRSK','VRSN','VRTX','WBD','XEL','ZM','ZS'
    ]
    sox_extra = [
        'ACLS','ADI','AEHR','AMAT','AMKR','ASML','COHU',
        'ENTG','ENVX','GFS','IPGP','KLAC','LRCX','MCHP',
        'MKSI','MPWR','MRVL','ONTO','POWI','QRVO','RMBS','SITM',
        'SWKS','SYNA','TER','TSM','UCTT','WOLF'
    ]
    todos_extra = list(set(nasdaq100_extra + sox_extra))
    ya_incluidos = set(pt + sa)
    for tk in todos_extra:
        if tk not in ya_incluidos:
            if tk in sox_extra:
                sbs.setdefault('Semiconductores SOX', []).append(tk)
            else:
                sbs.setdefault('Nasdaq 100', []).append(tk)
    sa = list(set(t for grp in sbs.values() for t in grp))
    print('  Ampliado con Nasdaq 100 y SOX: ' + str(len(sa)) + ' valores totales')

    print('▸ Descargando ' + str(len(sa)) + ' valores (S&P 500 + Nasdaq 100 + SOX)...')
    cs, vs, hs, ls = download_prices(sa, period='1y')
    sr  = analyze_universe(sbs, bs, cs, vs, hs, ls, spy_healthy=spy_ok, spy_score=spy_score_continuo)
    sgs = calc_groups(sr, is_sp=True)
    print('  OK ' + str(len(sr)) + ' valores')
    faltan_uni = sorted(set(sa) - {r['ticker'] for r in sr})  # PUNTO 26
    if faltan_uni: print('  Sin datos/analisis en universo: ' + ', '.join(faltan_uni))

    ar = pr + sr
    ts = datetime.now(madrid).strftime('%d/%m/%Y %H:%M')
    all_groups = sorted(pgs + sgs, key=lambda x: x['score'], reverse=True)

    # PUNTO 8 — Amplitud de mercado (informativa, sobre el universo amplio S&P500+Nasdaq100+SOX)
    print('\n▸ Calculando amplitud de mercado...')
    breadth = calc_market_breadth(cs, bs)
    breadth = anotar_tendencia_en_breadth(breadth, macro, ts)   # P37b — espejo manual de main()
    if breadth.get('pct_sobre_mm50') is not None:
        print('  ' + str(breadth.get('pct_sobre_mm20','?')) + '% sobre MM20 | ' + str(breadth['pct_sobre_mm50']) + '% sobre MM50 | ' + str(breadth['pct_sobre_mm200']) +
              '% sobre MM200 | Avance/Descenso: ' + str(breadth['avance']) + '/' + str(breadth['descenso']) +
              ' | Nuevos max/min 52s: ' + str(breadth['nuevos_max_52s']) + '/' + str(breadth['nuevos_min_52s']) +
              ' | McClellan: ' + str(breadth.get('mcclellan','?')) + ' | MM200 SPY: ' + str(breadth['pendiente_mm200_spy']))
    else:
        print('  Amplitud no disponible (datos insuficientes)')

    # Obtener fundamentales de valores con setup valido
    print('\n▸ Obteniendo datos fundamentales...')
    valid_tickers = list(set(
        v['ticker'] for v in sorted(ar, key=lambda x: x['score'] or 0, reverse=True)
        if v.get('entry_range', {}).get('entry_lo')
        and (v.get('entry_range', {}).get('rr') or 0) >= 2.0
        and (v.get('adx') or 0) >= 20
        and v.get('riesgo') != 'ALTO'
    ))[:20]
    fundamentales = get_fundamentals(valid_tickers) if valid_tickers else {}
    print('  OK ' + str(len(fundamentales)) + ' valores con datos fundamentales')

    # PUNTO 10 — update_setups_history ANTES de generate_analysis
    print('\n▸ Actualizando historico de setups...')
    history = update_history(all_groups, all_values=sorted(ar, key=lambda x: x['score'] or 0, reverse=True))
    values_sorted = sorted(ar, key=lambda x: x['score'] or 0, reverse=True)
    setups_history, evaluaciones = update_setups_history(values_sorted, all_groups)
    upload_to_github('setups_history.json', json.dumps(clean_nan(setups_history), ensure_ascii=False))
    if evaluaciones:
        chk = [e for e in evaluaciones if e.get('checkpoint')]
        n_ok = sum(1 for e in chk if e['resultado'] == 'target')
        n_stop = sum(1 for e in chk if e['resultado'] == 'stop')
        n_ab = sum(1 for e in chk if e['resultado'] == 'abierto')
        print('  Evaluaciones diarias: ' + str(len(evaluaciones)) + ' | Checkpoints 5/10/20: ' + str(len(chk)) + ' (Target: ' + str(n_ok) + ' | Stop: ' + str(n_stop) + ' | Abierto: ' + str(n_ab) + ')')
        # PUNTO 40 — espejo manual: resoluciones primero (un stop es mas urgente que una alerta).
        _resoluciones = resoluciones_por_ticker(evaluaciones)
        if _resoluciones:
            print(f'  🎯 SETUPS RESUELTOS ({len(_resoluciones)}): stop u objetivo alcanzado')
            for _r in _resoluciones:
                _que = 'STOP' if _r['resultado'] == 'stop' else 'OBJETIVO'
                print(f'     {_r["ticker"]}: setup del {_r["fecha_setup"]} | {_que} alcanzado | '
                      f'entrada ${_r["precio_entrada"]} -> ${_r["precio_actual"]} | ret={_r["ret_pct"]}%')
        alertas_cmf = dedup_alertas_por_ticker([e for e in evaluaciones if e.get('alerta_cmf')])
        alertas_st = dedup_alertas_por_ticker([e for e in evaluaciones if e.get('alerta_supertrend')])
        pre_rotos = [e for e in evaluaciones if e.get('supertrend_pre_roto')]
        if alertas_st:
            print('  ALERTAS SUPERTREND (' + str(len(alertas_st)) + ' setups con ruptura posterior a la entrada):')
            for a in alertas_st:
                d = a.get('supertrend_dias')
                etiq = 'ruptura NUEVA (esta sesion)' if (d is None or d <= 1) else 'ruptura persistente (' + str(d) + ' sesiones)'
                print('     ' + a['ticker'] + ': setup del ' + a['fecha_setup'] + ' | Supertrend bajista, nivel=$' + str(a['supertrend_nivel']) + ', ' + etiq + ' | ret=' + str(a['ret_pct']) + '% | Senial de salida total')
        if pre_rotos:
            print('  INFO: ' + str(len(set(a['ticker'] for a in pre_rotos))) + ' tickers con Supertrend ya bajista AL CREARSE (sin alerta; supertrend_pre_roto=true en data.json): ' + ', '.join(sorted(set(a['ticker'] for a in pre_rotos))))
        cmf_pre = [e for e in evaluaciones if e.get('cmf_pre_negativo')]
        if cmf_pre:
            print('  INFO: ' + str(len(set(a['ticker'] for a in cmf_pre))) + ' tickers con CMF ya negativo AL CREARSE (sin alerta blanda; cmf_pre_negativo=true en data.json): ' + ', '.join(sorted(set(a['ticker'] for a in cmf_pre))))
        if alertas_cmf:
            print('  ALERTAS CMF (' + str(len(alertas_cmf)) + ' setups con distribucion confirmada, CMF<0 en 3+ sesiones):')
            for a in alertas_cmf:
                print('     ' + a['ticker'] + ': setup del ' + a['fecha_setup'] + ' | CMF negativo ' + str(a.get('cmf_dias_negativo')) + ' sesiones (actual=' + str(a['cmf_actual']) + ') | ret=' + str(a['ret_pct']) + '% | Considerar reducir posicion 50%')

    print('\n▸ Generando analisis Claude...')
    # P43 — espejo manual: anotar resoluciones con su fecha de primera deteccion.
    resoluciones_anotadas = anotar_resoluciones(evaluaciones, macro, ts)
    data_tmp = {'timestamp': ts, 'mode': 'S&P500 + Watchlist', 'groups': all_groups,
                'values': sorted(ar, key=lambda x: x['score'] or 0, reverse=True),
                'spy_healthy': spy_ok, 'macro': macro, 'fundamentales': fundamentales,
                'breadth': breadth, 'evaluaciones': evaluaciones, 'resoluciones': resoluciones_anotadas}
    try:
        analisis = generate_analysis(data_tmp, ANTHROPIC_KEY)
        print('  OK')
    except Exception as e:
        print('  Aviso: ' + str(e))
        analisis = 'Analisis no disponible.'
    # PUNTO 36 — validar integridad del informe generado
    # ARREGLO (02/08/2026): antes pasaba 'valid', local de generate_analysis — la
    # comprobacion de tickers nunca llegaba a ejecutarse. Espejo manual de main().
    _valido, _avisos = validar_informe(analisis, ar)
    # PUNTO 55 — espejo manual: conservar el analisis anterior si el de hoy vino vacio.
    analisis, _rescatado, _no_publicar_data = rescatar_analisis_anterior(analisis, _avisos)
    if _avisos:
        for av in _avisos: print('  AVISO informe: ' + av)

    data = {
        'timestamp': ts, 'mode': 'S&P500 + Watchlist',
        'groups': all_groups,
        'values': values_sorted,
        'analisis': analisis,
        'spy_healthy': spy_ok,
        'evaluaciones': evaluaciones,
        'resoluciones': resoluciones_anotadas,
        'spy_ret_1w': spy_ret_1w,
        'spy_ret_1m': spy_ret_1m,
        'macro': macro,
        'breadth': breadth,
        'fundamentales': fundamentales,
    }

    # PUNTO 27 (19/07/2026) — la Celda 4 ejecutaba el pipeline completo SIN alertas Telegram
    # (la llamada vivia solo en la Celda 3 antigua, eliminada el 06/08/2026): semanas de silencio.
    print('\n▸ Generando alertas Telegram...')
    generate_alerts(data, history, spy_ok, bench_series=bs)

    print('\n▸ Subiendo a GitHub...')
    # PUNTO 24 — persistir la serie diaria de amplitud (dataset de la calibracion de septiembre)
    breadth_history = actualizar_breadth_history(breadth, macro, ts)
    ficheros_subida = {
        'data.json':           json.dumps(data,           ensure_ascii=False),
        'history.json':        json.dumps(history,        ensure_ascii=False),
        'setups_history.json': json.dumps(setups_history, ensure_ascii=False),
    }
    if breadth_history is not None:
        ficheros_subida['breadth_history.json'] = json.dumps(breadth_history, ensure_ascii=False)
    # CURVA DE TIPOS (01/08) — espejo manual del bloque de main() en scanner.py.
    # FUERA de la verificacion AST: si se toca alli, hay que tocarlo aqui a mano.
    rates_history = actualizar_rates_history(macro, ts)
    if rates_history is not None:
        ficheros_subida['rates_history.json'] = json.dumps(rates_history, ensure_ascii=False)
    # PUNTO 39 — espejo manual: historico permanente de checkpoints.
    checkpoints_history = actualizar_checkpoints_history(evaluaciones, macro, ts)
    if checkpoints_history is not None:
        ficheros_subida['checkpoints_history.json'] = json.dumps(checkpoints_history, ensure_ascii=False)
    # PUNTO 43 — espejo manual: historico permanente de resoluciones.
    resoluciones_history = actualizar_resoluciones_history(evaluaciones, macro, ts)
    if resoluciones_history is not None:
        ficheros_subida['resoluciones_history.json'] = json.dumps(resoluciones_history, ensure_ascii=False)
    # PUNTO 56 — espejo manual: serie diaria de put/call.
    putcall_history = actualizar_putcall_history(putcall, macro, ts)
    if putcall_history is not None:
        ficheros_subida['putcall_history.json'] = json.dumps(putcall_history, ensure_ascii=False)
    # PUNTO 68 — espejo manual: historico de alertas con su desenlace.
    alertas_history = actualizar_alertas_history(evaluaciones, macro, ts)
    if alertas_history is not None:
        ficheros_subida['alertas_history.json'] = json.dumps(alertas_history, ensure_ascii=False)
    # PUNTO 75 — registro permanente de setups: una fila por setup creado, para siempre.
    # setups_history[-1] es la entrada de HOY (update_setups_history la añade al final).
    _setups_hoy = (setups_history[-1].get('setups') if setups_history else None) or []
    setups_registro = actualizar_setups_registro(_setups_hoy, values_sorted, macro, ts)
    if setups_registro is not None:
        ficheros_subida['setups_registro.json'] = json.dumps(setups_registro, ensure_ascii=False)
    # P57 — espejo manual: serie diaria del spy_health score continuo.
    try:
        _spy_close_val = float(bs.iloc[-1]) if bs is not None and len(bs) > 0 else None
        _ma200_val = float(bs.rolling(200).mean().iloc[-1]) if bs is not None and len(bs) >= 200 else None
    except Exception:
        _spy_close_val = None; _ma200_val = None
    spy_health_history = actualizar_spy_health_history(spy_score_continuo, spy_ok, _spy_close_val, _ma200_val, ts)
    if spy_health_history is not None:
        ficheros_subida['spy_health_history.json'] = json.dumps(clean_nan(spy_health_history), ensure_ascii=False)
    # P65 — no pisar data.json con un vuelco incompleto (ver data_json_publicable).
    _publicable, _motivo = data_json_publicable()
    if not _publicable:
        ficheros_subida.pop('data.json', None)
        print(f'  🔴 {_motivo}')
    # P78 — tampoco pisarlo con el informe vacio cuando no se ha podido comprobar que hay
    # al otro lado. El resto de ficheros suben igual: tienen dedupe por fecha.
    if _no_publicar_data and 'data.json' in ficheros_subida:
        ficheros_subida.pop('data.json', None)
        print('  🔴 data.json NO SE SUBE: el informe vino vacio y la lectura del anterior '
              'fallo, asi que se conserva intacto lo que hubiera en el repo.')
    upload_files_to_github(ficheros_subida)

    todos = all_groups
    print('\n=== TOP 5 TEMAS ===')
    # P63 — espejo: los temas con menos de MIN_MIEMBROS_TEMA valores van aparte.
    _comparables = [g for g in todos if g.get('tema_comparable')]
    _no_comp = [g for g in todos if not g.get('tema_comparable')]
    for i, g in enumerate(_comparables[:5]):
        print('#' + str(i+1) + '  ' + str(g['score']) + '  ' + g['group'])
    if _no_comp:
        print('(fuera de ranking, menos de ' + str(MIN_MIEMBROS_TEMA) + ' valores: '
              + ', '.join(g['group'] + ' ' + str(g['score']) + ' n=' + str(g['n']) for g in _no_comp[:5]) + ')')
    bos = [r for r in ar if r['breakout']]
    print('Rupturas: ' + str(len(bos)))
    print('SPY saludable: ' + str(spy_ok))
    print('\nWeb: https://Carolo-III.github.io/scanner-temas')
